# Create Work Volumes (oxjob #1554)

Which book each Crossref chapter is in, and which proceedings volume each Crossref proceedings paper is in (#1528,
21 Nov 2026). A chapter is a work and its book is a work; `CreateWorksBase` puts the book on the chapter as
`volume {id, display_name}`, and `CreateWorkParents` turns it into `parent_works` with relationship `volume`.

```
openalex.crossref.crossref_deduplicated (raw: ISBNs, types)  --> crossref_volume_records
crossref_volume_records + location_work_ids                  --> work_volumes (one row per chapter location)
work_volumes                                                 --> CreateWorksBase.volume
```

Runs after `Map_Work_Ids` (both ends need work ids) and before `Works_Base`. Sizing and rule precision:
oxjobs `working/link-book-chapters-to-books/` (EXPLORE.md, DESIGN.md).

### Crossref book records and their parts, ISBNs normalized

In [ ]:
-- Crossref book records and their parts, with ISBNs normalized to ISBN-13 (oxjob #1554).
-- Walden's Crossref parse drops ISBNs, so they are read here from the raw feed. One row per Crossref record of a
-- book-ish type: containers (the books) and parts (chapters, sections, entries). Read by the next cell and by
-- CreateLocationsWithTypes (cr_type, title). oxjob #1528 (21 Nov 2026): proceedings volumes are containers and
-- proceedings papers are parts; container_title, publisher and year serve the proceedings title rule, and since
-- 22 Nov container_title2 (the second container title: the own title of the volume when the first names the series).
-- Also since 22 Nov, for minted proceedings volumes (cells below): issns (well-formed ISSNs, uppercased, sorted),
-- volume_number (the deposited volume with any Vol. prefix and leading zeros dropped) and event_name.
-- oxjob #1528 (24 Nov 2026): title_year is year, except for a proceedings paper Crossref gives no issued year (~850K,
-- most IEEE papers before 2007, ASEE, ISMRM): the first year (at most next year) that its first container title, its
-- second container title, its event name or its DOI (10.1109/icc.2005.1494724) names, in that order; the DOI comes
-- last because an old IEEE DOI year can be the digitization year. Only the proceedings title rule and the phase 2
-- minted volumes read it; the DOI-stem rule and phase 1 keep year (an extra year would tip the stem-series guard).
CREATE OR REPLACE TABLE openalex.works.crossref_volume_records
CLUSTER BY (doi)
AS
WITH base AS (
  SELECT
    LOWER(native_id) AS doi,
    type AS cr_type,
    GET(title, 0) AS title,
    GET(`container-title`, 0) AS container_title,
    GET(`container-title`, 1) AS container_title2,
    ARRAY_SORT(ARRAY_DISTINCT(FILTER(TRANSFORM(ISSN, x -> UPPER(TRIM(x))), x -> x RLIKE '^[0-9]{4}-[0-9]{3}[0-9X]$'))) AS issns,
    NULLIF(REGEXP_REPLACE(REGEXP_REPLACE(LOWER(TRIM(volume)), '^(vol(ume)?[.]?\\s*)', ''), '^0+(?=[0-9])', ''), '') AS volume_number,
    NULLIF(TRIM(event.name), '') AS event_name,
    publisher,
    GET(GET(issued.`date-parts`, 0), 0) AS year,
    ARRAY_DISTINCT(FILTER(TRANSFORM(ISBN, x -> REGEXP_REPLACE(UPPER(x), '[^0-9X]', '')), s -> LENGTH(s) IN (10, 13))) AS raw_isbns
  FROM openalex.crossref.crossref_deduplicated
  WHERE type IN (
    -- containers
    'book', 'monograph', 'edited-book', 'reference-book', 'book-set', 'proceedings',
    -- parts
    'book-chapter', 'book-section', 'book-part', 'book-track', 'reference-entry', 'other', 'proceedings-article'
  )
)
SELECT
  doi,
  cr_type,
  cr_type IN ('book', 'monograph', 'edited-book', 'reference-book', 'book-set', 'proceedings') AS is_container,
  title,
  container_title,
  container_title2,
  issns,
  volume_number,
  event_name,
  publisher,
  year,
  CASE WHEN year IS NULL AND cr_type = 'proceedings-article' THEN AGGREGATE(ARRAY(
    TRY_CAST(REGEXP_EXTRACT(COALESCE(container_title, ''), '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])', 1) AS BIGINT),
    TRY_CAST(REGEXP_EXTRACT(COALESCE(container_title2, ''), '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])', 1) AS BIGINT),
    TRY_CAST(REGEXP_EXTRACT(COALESCE(event_name, ''), '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])', 1) AS BIGINT),
    TRY_CAST(REGEXP_EXTRACT(doi, '^10\\.[0-9]+/[^/]*?[._-](19[5-9][0-9]|20[0-3][0-9])[._-]', 1) AS BIGINT)),
    CAST(NULL AS BIGINT), (acc, y) -> COALESCE(acc, CASE WHEN y <= YEAR(CURRENT_DATE()) + 1 THEN y END))
  ELSE year END AS title_year,
  -- ISBN-10 -> ISBN-13: prefix 978, drop the old check digit, recompute the EAN-13 one
  ARRAY_DISTINCT(FILTER(TRANSFORM(raw_isbns, s -> CASE
    WHEN LENGTH(s) = 13 THEN s
    WHEN LENGTH(s) = 10 THEN CONCAT('978', SUBSTR(s, 1, 9), CAST((10 - AGGREGATE(SEQUENCE(1, 12), 0,
      (acc, i) -> acc + CAST(SUBSTR(CONCAT('978', SUBSTR(s, 1, 9)), i, 1) AS INT) * (CASE WHEN i % 2 = 1 THEN 1 ELSE 3 END)) % 10) % 10 AS STRING))
  END), s -> s RLIKE '^97[89][0-9]{10}$')) AS isbns
FROM base

### Link each part to its book's work

In [ ]:
-- work_volumes (oxjob #1554): which book each Crossref chapter is in, as OpenAlex work ids; since oxjob #1528 (21 Nov
-- 2026) also which proceedings volume each Crossref proceedings paper is in.
-- A part (chapter, section, entry, proceedings paper) links to a Crossref book or proceedings record by, in order of trust:
--   doi_stem     the part's DOI minus its last -_. segment, or cut at the first -_. after the slash, is a book record
--                (10.4324/9780429440229-7 -> 10.4324/9780429440229; 10.1007/978-3-642-40760-4_49 -> ...-4)
--   isbn         a part ISBN names exactly one book record, else exactly one with the part's DOI prefix
--   doi_pattern  Oxford Academic-hosted presses (...003.NNNN -> ...001.0001) and Peter Lang (10.3726/<isbn>/N)
--   container_title  proceedings papers only: the paper's container title and publisher equal a Crossref proceedings
--                record's title and publisher, that pair names exactly one proceedings record, and the two years are at
--                most one apart (a series title reused every year names one year's volume only)
--   container_title_2  the same with the paper's second container title, used only when no other rule names a volume:
--                series that deposit the series as container-title[0] and the volume's title as [1] (SEG, WIT, LREC)
--   container_title_undated  either title rule for a proceedings paper Crossref gives no issued year, on its title_year
--                (the year its container titles, event or DOI name; oxjob #1528, 24 Nov 2026), used only when no DOI or
--                ISBN rule names a volume: the same year first, one year apart only when the matched title itself names
--                a year (a yearless series title registered for one year must not take the next edition), and never for
--                a work that another rule already links (so no work changes its volume)
-- When the stem and the ISBN name different books, no link (a set's or series' ISBN on the part, or a junk stem);
-- likewise when the container title names a different volume than a DOI rule.
-- Proceedings papers take a DOI stem only when the years are at most one apart, and never to a proceedings record that
-- papers from more than three years reach by stem: that is a series record, not a volume (10.5151/2594-5327).
-- Book parts never link to proceedings records, so their links are what they were before proceedings were added.
-- Both ends are mapped through location_work_ids, so the book is the work its own Crossref record lands on (that
-- settles book DOIs that two works share). One row per part location; CreateWorksBase picks the primary location's.
CREATE OR REPLACE TABLE openalex.works.work_volumes
CLUSTER BY (work_id)
AS
WITH containers AS (
  SELECT doi, cr_type, title, publisher, year, isbns FROM openalex.works.crossref_volume_records WHERE is_container
),
parts AS (
  SELECT doi, cr_type, isbns, LOWER(TRIM(container_title)) AS container_key,
    LOWER(TRIM(container_title2)) AS container_key2, publisher, year, title_year,
    REGEXP_EXTRACT(doi, '^(.*)[-_.][^-_./]+$', 1) AS stem1,
    REGEXP_EXTRACT(doi, '^(.*?)[-_.][^/]*$', 1) AS stem2,
    CASE
      WHEN doi RLIKE '\\.003\\.[0-9]{4}$' THEN REGEXP_REPLACE(doi, '\\.003\\.[0-9]{4}$', '.001.0001')
      WHEN doi RLIKE '^10\\.3726/[^/]+/[0-9]+$' THEN REGEXP_REPLACE(doi, '/[0-9]+$', '')
    END AS pattern_key
  FROM openalex.works.crossref_volume_records
  WHERE NOT is_container
),
isbn_index AS (
  SELECT isbn, COLLECT_SET(doi) AS book_dois
  FROM (SELECT doi, EXPLODE(isbns) AS isbn FROM containers WHERE cr_type <> 'proceedings')
  GROUP BY isbn
),
isbn_candidates AS (
  SELECT p.doi, ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(i.book_dois))) AS book_dois
  FROM (SELECT doi, EXPLODE(isbns) AS isbn FROM parts) p
  JOIN isbn_index i ON i.isbn = p.isbn
  GROUP BY p.doi
),
stem_series AS (  -- proceedings records that proceedings papers from more than three years reach by DOI stem
  SELECT COALESCE(s1.doi, s2.doi) AS book_doi
  FROM parts p
  LEFT JOIN containers s1 ON s1.doi = p.stem1 AND s1.cr_type = 'proceedings'
  LEFT JOIN containers s2 ON s2.doi = p.stem2 AND p.stem2 <> p.stem1 AND s2.cr_type = 'proceedings'
  WHERE p.cr_type = 'proceedings-article' AND COALESCE(s1.doi, s2.doi) IS NOT NULL
  GROUP BY 1
  HAVING COUNT(DISTINCT p.year) > 3
),
title_index AS (  -- proceedings records whose (title, publisher) names exactly one proceedings record
  SELECT LOWER(TRIM(title)) AS title_key, publisher, MIN(doi) AS book_doi, MIN(year) AS book_year
  FROM containers
  WHERE cr_type = 'proceedings' AND TRIM(COALESCE(title, '')) <> '' AND publisher IS NOT NULL
  GROUP BY 1, 2
  HAVING COUNT(*) = 1
),
picks AS (
  SELECT
    p.doi,
    CASE
      WHEN p.cr_type = 'proceedings-article' AND (ss.book_doi IS NOT NULL
        OR ABS(p.year - CASE WHEN s1.doi IS NOT NULL THEN s1.year ELSE s2.year END) > 1) THEN NULL
      ELSE COALESCE(s1.doi, s2.doi)
    END AS stem_book,
    CASE
      WHEN SIZE(c.book_dois) = 1 THEN c.book_dois[0]
      WHEN SIZE(FILTER(c.book_dois, b -> SUBSTRING_INDEX(b, '/', 1) = SUBSTRING_INDEX(p.doi, '/', 1))) = 1
        THEN FILTER(c.book_dois, b -> SUBSTRING_INDEX(b, '/', 1) = SUBSTRING_INDEX(p.doi, '/', 1))[0]
    END AS isbn_book,
    pt.doi AS pattern_book,
    CASE WHEN p.cr_type = 'proceedings-article' AND ABS(p.year - ti.book_year) <= 1 THEN ti.book_doi END AS title_book1,
    CASE WHEN p.cr_type = 'proceedings-article' AND ABS(p.year - ti2.book_year) <= 1 THEN ti2.book_doi END AS title_book2,
    CASE WHEN p.cr_type = 'proceedings-article' AND p.year IS NULL THEN CASE
      WHEN p.title_year = ti.book_year THEN ti.book_doi
      WHEN p.title_year = ti2.book_year THEN ti2.book_doi
      WHEN ABS(p.title_year - ti.book_year) = 1 AND ti.title_key RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN ti.book_doi
      WHEN ABS(p.title_year - ti2.book_year) = 1 AND ti2.title_key RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN ti2.book_doi
    END END AS title_book_undated
  FROM parts p
  LEFT JOIN containers s1 ON s1.doi = p.stem1 AND (p.cr_type = 'proceedings-article' OR s1.cr_type <> 'proceedings')
  LEFT JOIN containers s2 ON s2.doi = p.stem2 AND p.stem2 <> p.stem1
    AND (p.cr_type = 'proceedings-article' OR s2.cr_type <> 'proceedings')
  LEFT JOIN stem_series ss ON ss.book_doi = COALESCE(s1.doi, s2.doi)
  LEFT JOIN isbn_candidates c ON c.doi = p.doi
  LEFT JOIN containers pt ON pt.doi = p.pattern_key AND (p.cr_type = 'proceedings-article' OR pt.cr_type <> 'proceedings')
  LEFT JOIN title_index ti ON ti.title_key = p.container_key AND ti.publisher = p.publisher
  LEFT JOIN title_index ti2 ON ti2.title_key = p.container_key2 AND ti2.publisher = p.publisher
),
picks2 AS (
  SELECT *, COALESCE(title_book1,
    CASE WHEN COALESCE(stem_book, isbn_book, pattern_book) IS NULL THEN COALESCE(title_book2, title_book_undated) END)
    AS title_book
  FROM picks
),
links AS (
  SELECT doi,
    COALESCE(stem_book, isbn_book, pattern_book, title_book) AS book_doi,
    CASE
      WHEN stem_book IS NOT NULL AND isbn_book IS NOT NULL THEN 'isbn_and_doi_stem'
      WHEN stem_book IS NOT NULL THEN 'doi_stem'
      WHEN isbn_book IS NOT NULL THEN 'isbn'
      WHEN pattern_book IS NOT NULL THEN 'doi_pattern'
      WHEN title_book1 IS NOT NULL THEN 'container_title'
      WHEN title_book2 IS NOT NULL THEN 'container_title_2'
      ELSE 'container_title_undated'
    END AS method
  FROM picks2
  WHERE COALESCE(stem_book, isbn_book, pattern_book, title_book) IS NOT NULL
    AND NOT (stem_book IS NOT NULL AND isbn_book IS NOT NULL AND stem_book <> isbn_book)
    AND NOT (title_book IS NOT NULL AND COALESCE(stem_book, isbn_book, pattern_book) IS NOT NULL
             AND title_book <> COALESCE(stem_book, isbn_book, pattern_book))
),
mapped AS (
SELECT
  part.work_id,
  l.doi AS native_id,
  book.work_id AS volume_work_id,
  l.book_doi AS volume_doi,
  c.title AS volume_display_name,
  l.method
FROM links l
JOIN openalex.works.location_work_ids part
  ON part.provenance = 'crossref' AND part.native_id_namespace = 'doi' AND part.native_id = l.doi
JOIN openalex.works.location_work_ids book
  ON book.provenance = 'crossref' AND book.native_id_namespace = 'doi' AND book.native_id = l.book_doi
JOIN containers c ON c.doi = l.book_doi
WHERE part.work_id IS NOT NULL AND book.work_id IS NOT NULL AND part.work_id <> book.work_id
)
SELECT m.* FROM mapped m
LEFT ANTI JOIN (SELECT DISTINCT work_id FROM mapped WHERE method <> 'container_title_undated') o
  ON m.method = 'container_title_undated' AND o.work_id = m.work_id

### Minted proceedings volumes (oxjob #1528, phase 1)

Crossref has no record for the volume of ~3.4M proceedings papers. Phase 1 mints one `book` work per volume that the
papers name by two permanent identifiers, the series ISSN and the volume number (Proc. SPIE 10000, AIP Conference
Proceedings 2220). The registry `openalex.works.minted_volumes` keeps every volume once minted, so a threshold that
drifts never deletes one; `CreateLocationsWithTypes` writes one location per registry row (provenance `openalex-series`,
namespace `volume`) the next night, `Map_Work_Ids` gives it a W id, and the last cell links the papers to it.

Registry `status`: `active` = minted and linked; `unlinked` = minted (the W id lives) but no papers link to it (use it
to pull links back without deleting ids); `excluded` = never minted (deposit errors found by the bar).

In [0]:
-- oxjob #1528 (22 Nov 2026): the registry of minted proceedings volumes, one row per (series, volume number). Rows are
-- only added or refreshed, never deleted: the W id of a minted volume must not die when a threshold below drifts.
CREATE TABLE IF NOT EXISTS openalex.works.minted_volumes (
  mint_id STRING COMMENT 'proceedings:<series issn, lowercased>:<volume number>, the native_id of the minted location',
  issn STRING COMMENT 'the series: its ISSN-L when the source registry knows it, else its lowest ISSN, fixed at minting',
  series_issns ARRAY<STRING> COMMENT 'every ISSN seen on the series papers (only grows), so a paper matches whichever it carries',
  volume_number STRING,
  title STRING,
  year INT COMMENT 'the papers modal year',
  published_date DATE COMMENT 'earliest paper date in the modal year',
  publisher STRING,
  source_id BIGINT COMMENT 'the series source the papers sit under',
  status STRING COMMENT 'active, unlinked or excluded (see the notebook text)',
  papers BIGINT COMMENT 'qualifying papers last time the volume was assessed',
  first_seen DATE,
  last_seen DATE,
  updated_at DATE COMMENT 'last day the title, year, date, publisher or source changed'
) CLUSTER BY (mint_id)

In [0]:
-- oxjob #1528 (22 Nov 2026), minted proceedings volumes, phase 1: which volumes qualify tonight. Two outside reviews
-- (GPT, Gemini, 22 Nov) shaped these rules:
--   series    the series of a paper is the one ISSN-L its ISSNs map to in the source registry, else its lowest ISSN. A series
--             qualifies when none of its proceedings papers links to a Crossref volume record by the rules above and
--             >= 90% sit under a conference or book-series source (journal supplements stay out)
--   numbering a series that reuses volume numbers across years (> 5% of its numbers span > 2 years) is out, and a volume
--             number must hold a digit
--   volume    >= 2 papers, >= 95% of them within the modal year +-1 (the others are not linked), a non-empty title
--   title     the second container title when it names one volume number of the series, else the one event name when
--             it does, else <series>, Volume N. Every pick is the most frequent value, ties to the larger, so reruns agree
--   new only  a volume is new when no registry row has its volume number and shares an ISSN with it, and it is minted
--             only if no work OpenAlex already has carries its title and year +-1 (no duplicate). Registry rows are
--             refreshed while they still qualify and kept whatever tonight says.
-- Bar 22 Nov 2026 (oxjob #1528 PLAN.md): papers 148/150 (SPIE 50, AIP 49, other 49), volumes 99/100.
MERGE INTO openalex.works.minted_volumes r
USING (
  WITH linked AS (SELECT DISTINCT native_id FROM openalex.works.work_volumes WHERE COALESCE(method, '') <> 'minted'),
  issn_map AS (
    SELECT issn, MIN(issn_l) AS issn_l
    FROM (SELECT UPPER(TRIM(i)) AS issn, UPPER(TRIM(issn_l)) AS issn_l
          FROM openalex.sources.sources LATERAL VIEW EXPLODE(issns) e AS i WHERE issn_l IS NOT NULL)
    GROUP BY issn HAVING COUNT(DISTINCT issn_l) = 1
  ),
  loc AS (
    SELECT LOWER(native_id) AS doi, MAX(source_id) AS source_id, MIN(published_date) AS published_date
    FROM openalex.works.locations_w_types
    WHERE provenance = 'crossref' AND native_id_namespace = 'doi'
    GROUP BY 1
  ),
  p0 AS (
    SELECT c.doi, c.issns, c.volume_number AS vnum, c.year, c.publisher,
      NULLIF(TRIM(c.container_title), '') AS container_title, NULLIF(TRIM(c.container_title2), '') AS container_title2,
      c.event_name, l.source_id, l.published_date, s.type AS source_type, k.native_id IS NOT NULL AS is_linked
    FROM openalex.works.crossref_volume_records c
    LEFT JOIN linked k ON k.native_id = c.doi
    LEFT JOIN loc l ON l.doi = c.doi
    LEFT JOIN openalex.sources.sources s ON s.id = l.source_id
    WHERE c.cr_type = 'proceedings-article' AND SIZE(c.issns) > 0
  ),
  pm AS (
    SELECT x.doi, COLLECT_SET(m.issn_l) AS ls
    FROM (SELECT doi, EXPLODE(issns) AS i FROM p0) x LEFT JOIN issn_map m ON m.issn = x.i
    GROUP BY x.doi
  ),
  papers AS (
    SELECT p0.*, CASE WHEN SIZE(pm.ls) = 1 THEN pm.ls[0] ELSE ARRAY_MIN(p0.issns) END AS issn
    FROM p0 JOIN pm USING (doi)
  ),
  series AS (
    SELECT issn, ARRAY_SORT(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(issns)))) AS series_issns
    FROM papers GROUP BY issn
    HAVING COUNT_IF(is_linked) = 0 AND COUNT_IF(source_type IN ('conference', 'book series')) >= 0.9 * COUNT(*)
  ),
  cand AS (
    SELECT p.*, s.series_issns FROM papers p JOIN series s USING (issn)
    WHERE p.vnum RLIKE '^[0-9a-z][0-9a-z.-]{0,19}$' AND p.vnum RLIKE '[0-9]'
      AND p.year BETWEEN 1900 AND YEAR(CURRENT_DATE()) + 1
  ),
  reuse AS (
    SELECT issn FROM (SELECT issn, vnum, MAX(year) - MIN(year) AS span FROM cand GROUP BY 1, 2)
    GROUP BY issn HAVING SUM(CASE WHEN span > 2 THEN 1 ELSE 0 END) > 0.05 * COUNT(*)
  ),
  c2 AS (SELECT cand.* FROM cand LEFT ANTI JOIN reuse USING (issn)),
  ycount AS (SELECT issn, vnum, year, COUNT(*) AS k FROM c2 GROUP BY 1, 2, 3),
  modal AS (SELECT issn, vnum, MAX_BY(year, k * 10000 + year) AS my FROM ycount GROUP BY 1, 2),
  c3 AS (SELECT c2.*, m.my FROM c2 JOIN modal m USING (issn, vnum)),
  vt AS (
    SELECT issn, LOWER(container_title2) AS k FROM c3 WHERE container_title2 IS NOT NULL
    GROUP BY 1, 2 HAVING COUNT(DISTINCT vnum) = 1
  ),
  ev AS (
    SELECT issn, LOWER(event_name) AS k FROM c3 WHERE event_name IS NOT NULL
    GROUP BY 1, 2 HAVING COUNT(DISTINCT vnum) = 1
  ),
  c4 AS (
    SELECT c3.*, CASE WHEN vt.k IS NOT NULL THEN c3.container_title2 END AS vtitle,
      CASE WHEN ev.k IS NOT NULL THEN c3.event_name END AS etitle
    FROM c3
    LEFT JOIN vt ON vt.issn = c3.issn AND vt.k = LOWER(c3.container_title2)
    LEFT JOIN ev ON ev.issn = c3.issn AND ev.k = LOWER(c3.event_name)
  ),
  c5 AS (
    SELECT c4.*,
      COUNT(publisher) OVER (PARTITION BY issn, vnum, publisher) AS n_pub,
      COUNT(source_id) OVER (PARTITION BY issn, vnum, source_id) AS n_src,
      COUNT(container_title) OVER (PARTITION BY issn, vnum, container_title) AS n_ser,
      COUNT(vtitle) OVER (PARTITION BY issn, vnum, vtitle) AS n_vt,
      COUNT(etitle) OVER (PARTITION BY issn, vnum, etitle) AS n_et
    FROM c4
  ),
  vols AS (
    SELECT issn, vnum, MAX(series_issns) AS series_issns, COUNT(*) AS n, COUNT_IF(ABS(year - my) <= 1) AS n_in,
      MAX(my) AS year, MIN(CASE WHEN YEAR(published_date) = my THEN published_date END) AS published_date,
      MAX_BY(publisher, STRUCT(n_pub, publisher)) AS publisher,
      MAX_BY(source_id, STRUCT(n_src, source_id)) AS source_id,
      MAX_BY(container_title, STRUCT(n_ser, container_title)) AS series_title,
      MAX_BY(vtitle, STRUCT(n_vt, vtitle)) AS volume_title,
      COUNT(DISTINCT LOWER(etitle)) AS n_events,
      MAX_BY(etitle, STRUCT(n_et, etitle)) AS event_title
    FROM c5 GROUP BY issn, vnum
  ),
  ok AS (
    SELECT * FROM (
      SELECT CONCAT('proceedings:', LOWER(issn), ':', vnum) AS mint_id, issn, series_issns, vnum AS volume_number,
        NULLIF(TRIM(COALESCE(volume_title, CASE WHEN n_events = 1 THEN event_title END,
          CASE WHEN series_title IS NOT NULL THEN CONCAT(series_title, ', Volume ', vnum) END)), '') AS title,
        CAST(year AS INT) AS year, COALESCE(published_date, MAKE_DATE(year, 1, 1)) AS published_date,
        publisher, source_id, n_in AS papers
      FROM vols WHERE n_in >= 2 AND n_in >= 0.95 * n
    ) WHERE title IS NOT NULL
  ),
  -- oxjob #1528 (23 Nov 2026): a volume whose papers sit in a phase 2 edition (cell below) that qualified last night or
  -- tonight is not new.
  p2 AS (
    SELECT DISTINCT c.doi FROM openalex.works.crossref_volume_records c
    JOIN openalex.works.minted_volumes r ON r.status = 'active' AND r.mint_id = CONCAT('edition:', SPLIT(c.doi, '/')[0], ':', SUBSTR(SHA1(REGEXP_REPLACE(LOWER(CASE WHEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') WHEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') END), '[^\\p{L}\\p{N}]', '')), 1, 16))
     AND r.last_seen >= DATE_SUB(CURRENT_DATE(), 1) AND ABS(c.year - r.year) <= 1
    WHERE c.cr_type = 'proceedings-article' AND r.mint_id LIKE 'edition:%'
  ),
  taken AS (SELECT DISTINCT c3.issn, c3.vnum FROM c3 JOIN p2 USING (doi)),
  fresh AS (
    SELECT ok.mint_id FROM ok LEFT ANTI JOIN openalex.works.minted_volumes m
      ON m.volume_number = ok.volume_number AND ARRAYS_OVERLAP(m.series_issns, ok.series_issns)
    LEFT ANTI JOIN taken t ON t.issn = ok.issn AND t.vnum = ok.volume_number
  ),
  dup AS (
    SELECT DISTINCT ok.mint_id FROM ok JOIN fresh USING (mint_id) JOIN openalex.works.openalex_works w
      ON LOWER(TRIM(w.title)) = LOWER(ok.title) AND ABS(w.publication_year - ok.year) <= 1
     AND w.type IN ('book', 'paratext', 'other', 'conference-paper', 'report')
  )
  SELECT ok.*, fresh.mint_id IS NOT NULL AS is_fresh, dup.mint_id IS NOT NULL AS is_dup
  FROM ok LEFT JOIN fresh USING (mint_id) LEFT JOIN dup USING (mint_id)
) s
ON r.mint_id = s.mint_id
WHEN MATCHED AND NOT (r.title <=> s.title AND r.year <=> s.year AND r.published_date <=> s.published_date
    AND r.publisher <=> s.publisher AND r.source_id <=> s.source_id) THEN UPDATE SET
  title = s.title, year = s.year, published_date = s.published_date, publisher = s.publisher, source_id = s.source_id,
  series_issns = ARRAY_SORT(ARRAY_DISTINCT(CONCAT(COALESCE(r.series_issns, ARRAY()), s.series_issns))), papers = s.papers,
  last_seen = CURRENT_DATE(), updated_at = CURRENT_DATE()
WHEN MATCHED THEN UPDATE SET
  series_issns = ARRAY_SORT(ARRAY_DISTINCT(CONCAT(COALESCE(r.series_issns, ARRAY()), s.series_issns))), papers = s.papers,
  last_seen = CURRENT_DATE()
WHEN NOT MATCHED AND s.is_fresh AND NOT s.is_dup THEN INSERT (mint_id, issn, series_issns, volume_number, title, year,
  published_date, publisher, source_id, status, papers, first_seen, last_seen, updated_at)
  VALUES (s.mint_id, s.issn, s.series_issns, s.volume_number, s.title, s.year, s.published_date, s.publisher,
  s.source_id, 'active', s.papers, CURRENT_DATE(), CURRENT_DATE(), CURRENT_DATE())

### Minted proceedings volumes, phase 2: an edition title with its year (oxjob #1528)

Papers whose container title names one edition by its year ("Proceedings of the 2026 AERA Annual Meeting", "ICALEO 2007:
26th International Congress on Laser Materials Processing") and whose series deposits no volume number. One `book` work
per (DOI prefix, edition title), in the same registry as phase 1 (`mint_id` = `edition:<prefix>:<hash of the title>`,
`issn` and `volume_number` empty). A title with a year names one edition, so next year's papers carry a new title and
never move this one. Phase 1 papers and volumes stay where they are: an edition that any phase 1 paper sits in is left
alone. Papers link only through editions that qualified in the latest run (`last_seen` = the newest date among phase 2
rows): a registry row whose edition stops qualifying keeps its W id but loses its links until it qualifies again (to
see them: `mint_id LIKE 'edition:%' AND status = 'active' AND last_seen < CURRENT_DATE()` after a run).

In [0]:
-- oxjob #1528 (23 Nov 2026), minted proceedings volumes, phase 2: volumes named by an edition title with its year.
-- The edition title of a paper is its first container title when that holds a year (1950-2039), else its second when that
-- does, HTML entities decoded. The key is (DOI prefix, edition title with case, spaces and punctuation dropped). An
-- edition qualifies when, counting every proceedings paper that carries its key (linked ones and ones with no year too):
--   the work of no paper links to a Crossref volume record by the rules above, and no paper takes a phase 1 minted volume
--   the DOI prefix is not one that registers volume records (under half its proceedings papers link to one, all time
--   and, when it has 20 or more, over the last four years)
--   >= 90% of the papers sit under a conference or book-series source (journal supplements stay out)
--   >= 2 works, >= 95% of the papers dated within the modal year +-1, every year in the title within 1 of it
--   at most one volume number, one event name, one source and one other container title among the papers (two
--   conferences or two volumes sharing a title split), and the DOIs do not split into numbered volumes under one
--   year-stem (10.15376/frc.1981.1.x and 10.15376/frc.1981.2.x are two volumes)
--   not OSTI (10.2172), which deposits individual reports and slides under the meeting name
-- Title: the most frequent spelling. Skipped: a title all of whose words are generic (CONFERENCE 2023, 2024 Annual
-- Conference), a title naming a day of a meeting (Day 2 Tue, February 22, 2022) and a periodical issue title (Vol. 1,
-- No. 24 (2022)). Minted only if no work OpenAlex already has, no registry row and no other edition tonight carries
-- that title (case, spaces and punctuation aside) with a year +-1 (no duplicate). Registry rows are refreshed while
-- they qualify (last_seen = the date of this run, which the link cell requires, and a row that stops qualifying on a
-- same-day rerun loses that date) and kept whatever tonight says.
-- A paper with no issued year counts in the year its titles, event or DOI name (title_year, oxjob #1528, 24 Nov
-- 2026), and since that year often comes from the edition title itself, an edition is also skipped when more
-- than 5% of its undated papers whose DOI names a year sit 2+ years from the edition year.
-- Bar 23 Nov 2026 (oxjob #1528 PLAN.md): see the notebook text.
MERGE INTO openalex.works.minted_volumes r
USING (
  WITH linked AS (SELECT DISTINCT work_id FROM openalex.works.work_volumes WHERE COALESCE(method, '') <> 'minted'),
  p1 AS (
    SELECT DISTINCT c.doi
    FROM openalex.works.crossref_volume_records c
    JOIN openalex.works.minted_volumes r
      ON r.status = 'active' AND r.volume_number = c.volume_number AND ARRAYS_OVERLAP(r.series_issns, c.issns)
     AND ABS(c.year - r.year) <= 1
    WHERE c.cr_type = 'proceedings-article'
  ),
  loc AS (
    SELECT LOWER(native_id) AS doi, MAX(source_id) AS source_id, MIN(published_date) AS published_date
    FROM openalex.works.locations_w_types
    WHERE provenance = 'crossref' AND native_id_namespace = 'doi'
    GROUP BY 1
  ),
  p0 AS (
    SELECT c.doi, SPLIT(c.doi, '/')[0] AS prefix, c.title_year AS year, c.year IS NULL AS undated,
      TRY_CAST(REGEXP_EXTRACT(c.doi, '^10\\.[0-9]+/[^/]*?[._-](19[5-9][0-9]|20[0-3][0-9])[._-]', 1) AS BIGINT) AS doi_year, c.publisher, c.volume_number AS vnum,
      NULLIF(TRIM(c.event_name), '') AS event_name, CASE WHEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') WHEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') END AS etitle, CASE WHEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') WHEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') END AS other_title,
      CONCAT('edition:', SPLIT(c.doi, '/')[0], ':', SUBSTR(SHA1(REGEXP_REPLACE(LOWER(CASE WHEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') WHEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') END), '[^\\p{L}\\p{N}]', '')), 1, 16)) AS mint_id, w.work_id, l.source_id, l.published_date, s.type AS source_type,
      k.work_id IS NOT NULL AS is_linked, q.doi IS NOT NULL AS is_p1
    FROM openalex.works.crossref_volume_records c
    LEFT JOIN openalex.works.location_work_ids w
      ON w.provenance = 'crossref' AND w.native_id_namespace = 'doi' AND w.native_id = c.doi
    LEFT JOIN linked k ON k.work_id = w.work_id
    LEFT JOIN p1 q ON q.doi = c.doi
    LEFT JOIN loc l ON l.doi = c.doi
    LEFT JOIN openalex.sources.sources s ON s.id = l.source_id
    WHERE c.cr_type = 'proceedings-article'
  ),
  pfx AS (
    SELECT prefix FROM p0 GROUP BY prefix
    HAVING COUNT_IF(is_linked) < 0.5 * COUNT(*)
       AND (COUNT_IF(year >= YEAR(CURRENT_DATE()) - 3) < 20
            OR COUNT_IF(is_linked AND year >= YEAR(CURRENT_DATE()) - 3) < 0.5 * COUNT_IF(year >= YEAR(CURRENT_DATE()) - 3))
  ),
  e AS (
    SELECT p0.*, REGEXP_EXTRACT_ALL(etitle, '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])', 1) AS tyears,
      FILTER(SPLIT(TRIM(REGEXP_REPLACE(LOWER(etitle), '[^\\p{L}\\p{N}]+', ' ')), ' '),
             x -> NOT (x IN ('proceedings', 'proceeding', 'of', 'the', 'and', 'abstract', 'abstracts', 'book', 'books', 'conference', 'conferences', 'annual', 'meeting', 'international', 'paper', 'papers', 'technical', 'extended', 'program', 'programme', 'congress', 'symposium', 'workshop', 'in', 'on', 'for', 'volume', 'vol', 'part', 'parts', 'issue', 'no', 'edition', 'digest', 'collected', 'selected', 'full', 'presentations', 'posters', 'poster', 'session', 'sessions', 'exhibition', 'expo', 'forum', 'convention', 'colloquium', 'seminar', 'summit', 'national', 'world', 'general', 'assembly', 'society', 'joint', 'de', 'la', 'le', 'des', 'du', 'der', 'und', 'y', 'del', 'e', 'di', 'a', 'b', 'c', 'd') OR x RLIKE '^[0-9]+(st|nd|rd|th|e|er|eme)?$')) AS words
    FROM p0 JOIN pfx USING (prefix)
    WHERE etitle IS NOT NULL AND prefix <> '10.2172'
  ),
  vsplit AS (
    SELECT DISTINCT mint_id FROM (
      SELECT mint_id, head FROM (
        SELECT mint_id, REGEXP_EXTRACT(stem, '^(.*)[._/-]([^._/-]+)$', 1) AS head,
          REGEXP_EXTRACT(stem, '^(.*)[._/-]([^._/-]+)$', 2) AS tail
        FROM (SELECT mint_id, REGEXP_EXTRACT(doi, '^(.*)[._/-][^._/-]+$', 1) AS stem FROM e)
      )
      WHERE tail RLIKE '^([0-9]{1,4}|[ivx]{1,4})$' AND head RLIKE '(19|20)[0-9]{2}$'
      GROUP BY mint_id, head, tail HAVING COUNT(*) >= 2
    ) GROUP BY mint_id, head HAVING COUNT(*) >= 2
  ),
  ycount AS (
    SELECT mint_id, year, COUNT(*) AS k FROM e WHERE year BETWEEN 1950 AND YEAR(CURRENT_DATE()) + 1 GROUP BY 1, 2
  ),
  modal AS (SELECT mint_id, MAX_BY(year, k * 10000 + year) AS my FROM ycount GROUP BY 1),
  e2 AS (
    SELECT e.*, m.my, ABS(e.year - m.my) <= 1 AS in_year,
      COUNT(publisher) OVER (PARTITION BY mint_id, publisher) AS n_pub,
      COUNT(etitle) OVER (PARTITION BY mint_id, etitle) AS n_t,
      COUNT(event_name) OVER (PARTITION BY mint_id, event_name) AS n_e
    FROM e LEFT JOIN modal m USING (mint_id)
  ),
  g AS (
    SELECT mint_id, MAX(my) AS year, COUNT(*) AS n, COUNT_IF(is_linked) AS n_real, COUNT_IF(is_p1) AS n_p1,
      COUNT_IF(in_year) AS n_in, COUNT_IF(undated AND doi_year IS NOT NULL) AS n_dy,
      COUNT_IF(undated AND ABS(doi_year - my) > 1) AS n_dy_off, COUNT(DISTINCT CASE WHEN in_year THEN work_id END) AS n_works,
      FORALL(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(tyears))), y -> ABS(CAST(y AS INT) - MAX(my)) <= 1)
        AND ARRAY_MAX(TRANSFORM(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(tyears))), y -> CAST(y AS INT)))
          - ARRAY_MIN(TRANSFORM(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(tyears))), y -> CAST(y AS INT))) <= 1 AS title_year_ok,
      COUNT(DISTINCT vnum) AS n_vnum, COUNT(DISTINCT LOWER(event_name)) AS n_event, COUNT(DISTINCT source_id) AS n_src,
      COUNT(DISTINCT LOWER(other_title)) AS n_other,
      COUNT_IF(source_type IN ('conference', 'book series')) AS n_conf,
      MIN(SIZE(words)) AS n_words,
      MIN(CASE WHEN YEAR(published_date) = my THEN published_date END) AS published_date,
      MAX_BY(publisher, STRUCT(n_pub, publisher)) AS publisher,
      MAX(source_id) AS source_id,
      MAX_BY(etitle, STRUCT(n_t, etitle)) AS etitle,
      MAX_BY(event_name, STRUCT(n_e, event_name)) AS event_name
    FROM e2 GROUP BY mint_id
  ),
  ok AS (
    SELECT * FROM (
      SELECT mint_id, CAST(NULL AS STRING) AS issn, CAST(NULL AS ARRAY<STRING>) AS series_issns,
        CAST(NULL AS STRING) AS volume_number,
        etitle AS title,
        CAST(year AS INT) AS year, COALESCE(published_date, MAKE_DATE(year, 1, 1)) AS published_date,
        publisher, source_id, n_in AS papers
      FROM g
      WHERE n_real = 0 AND n_p1 = 0 AND n_conf >= 0.9 * n AND n_works >= 2 AND n_in >= 0.95 * n AND title_year_ok AND n_dy_off <= 0.05 * n_dy
        AND n_vnum <= 1 AND n_event <= 1 AND n_src <= 1 AND n_other <= 1 AND n_words > 0
        AND mint_id NOT IN (SELECT mint_id FROM vsplit)
    ) WHERE NULLIF(TRIM(title), '') IS NOT NULL
      AND NOT LOWER(title) RLIKE '^(session|track|week) *[0-9ivx]'
      AND NOT LOWER(title) RLIKE '(^|[^a-z])day *[0-9]{1,2}([^0-9]|$)'
      AND NOT LOWER(title) RLIKE '(mon|tue|tues|wed|wednes|thu|thur|thurs|fri|sat|satur|sun)(day)?,? ([0-9]{1,2} )?(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]* [0-9]'
      AND NOT LOWER(title) RLIKE 'vol(ume)?[.]? *[0-9]+[, ]+(no|issue|iss|num|number)[.]? *[0-9]+'
  ),
  same AS (
    SELECT tk FROM (SELECT REGEXP_REPLACE(LOWER(title), '[^\\p{L}\\p{N}]', '') AS tk, mint_id FROM ok)
    GROUP BY tk HAVING COUNT(DISTINCT mint_id) > 1
  ),
  fresh AS (SELECT ok.mint_id FROM ok LEFT ANTI JOIN openalex.works.minted_volumes m ON m.mint_id = ok.mint_id),
  known AS (
    SELECT REGEXP_REPLACE(LOWER(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '')), '[^\\p{L}\\p{N}]', '') AS tk, publication_year AS y
    FROM openalex.works.openalex_works
    WHERE type IN ('book', 'paratext', 'other', 'conference-paper', 'report') AND title IS NOT NULL
    UNION ALL
    SELECT REGEXP_REPLACE(LOWER(title), '[^\\p{L}\\p{N}]', ''), year FROM openalex.works.minted_volumes
  ),
  dup AS (
    SELECT DISTINCT f.mint_id
    FROM (SELECT ok.mint_id, ok.year, REGEXP_REPLACE(LOWER(ok.title), '[^\\p{L}\\p{N}]', '') AS tk
          FROM ok JOIN fresh USING (mint_id)) f
    JOIN known k ON k.tk = f.tk AND ABS(k.y - f.year) <= 1
  )
  SELECT ok.*, fresh.mint_id IS NOT NULL AS is_fresh, dup.mint_id IS NOT NULL OR sm.tk IS NOT NULL AS is_dup
  FROM ok LEFT JOIN fresh USING (mint_id) LEFT JOIN dup USING (mint_id)
  LEFT JOIN same sm ON sm.tk = REGEXP_REPLACE(LOWER(ok.title), '[^\\p{L}\\p{N}]', '')
) s
ON r.mint_id = s.mint_id
WHEN MATCHED AND NOT (r.title <=> s.title AND r.year <=> s.year AND r.published_date <=> s.published_date
    AND r.publisher <=> s.publisher AND r.source_id <=> s.source_id) THEN UPDATE SET
  title = s.title, year = s.year, published_date = s.published_date, publisher = s.publisher, source_id = s.source_id,
  papers = s.papers, last_seen = CURRENT_DATE(), updated_at = CURRENT_DATE()
WHEN MATCHED THEN UPDATE SET papers = s.papers, last_seen = CURRENT_DATE()
WHEN NOT MATCHED AND s.is_fresh AND NOT s.is_dup THEN INSERT (mint_id, issn, series_issns, volume_number, title, year,
  published_date, publisher, source_id, status, papers, first_seen, last_seen, updated_at)
  VALUES (s.mint_id, s.issn, s.series_issns, s.volume_number, s.title, s.year, s.published_date, s.publisher,
  s.source_id, 'active', s.papers, CURRENT_DATE(), CURRENT_DATE(), CURRENT_DATE())
WHEN NOT MATCHED BY SOURCE AND r.mint_id LIKE 'edition:%' AND r.last_seen = CURRENT_DATE() THEN UPDATE SET
  last_seen = DATE_SUB(CURRENT_DATE(), 1)

### Minted proceedings volumes, phase 3: numbered volumes of one meeting edition (oxjob #1528)

Some societies deposit a conference paper with the volume's own title as its container title ("Volume 6: Manufacturing
Materials & Metallurgy; ...") and the meeting edition as its event ("ASME Turbo Expo 2026: Turbomachinery Technical
Conference and Exposition"), and register no record for the volume. Phase 3 mints one `book` work per (DOI prefix, event,
volume title), titled "<event>, <volume title>", through the same registry (`mint_id` `event-volume:<prefix>:<hash>`).
It takes only papers with no deposited volume number and no year in a container title, so phases 1, 2 and 3 never claim
the same paper. Rollback as for phases 1 and 2: set the rows' status to `unlinked`.

In [0]:
-- oxjob #1528 (24 Nov 2026), minted proceedings volumes, phase 3: numbered volumes of one meeting edition. A paper takes
-- part when its container title names a volume number (Volume 6: ..., Vol. 2A), no container title names a year, it has
-- no deposited volume number and its event names one year; the guards below count every paper that carries the key. The key is (DOI prefix, event, volume title), entities
-- decoded, case, spaces and punctuation dropped. A volume qualifies when, counting every paper that carries its key:
--   the work of no paper links to a Crossref volume record by the rules above
--   the DOI prefix is not one that registers volume records (under half its proceedings papers link to one)
--   >= 90% of the papers sit under a conference or book-series source, >= 2 works, one source, one second title
--   >= 95% of the papers carry an issued year within 1 of the event year (the issued year is the independent check)
--   every paper carrying the key is a phase 3 paper (none has a volume number or a year in a container title)
--   the DOI prefix is on the validated list (ASME 10.1115; another publisher needs its own bar)
-- Title: the most frequent spelling of event (trailing punctuation dropped) and volume title, joined by a comma. Minted
-- only if no work OpenAlex already has, and no registry row, carries that title (or a book, report or other carries the
-- bare volume title, when it says more than its number) with the event year +-1, and no other volume tonight has it.
-- Registry rows are refreshed while they qualify (last_seen = the date of this run, read by the phase 3 link cell) and
-- kept whatever tonight says. Bar 24 Nov 2026 (oxjob #1528 PLAN.md): see the notebook text.
MERGE INTO openalex.works.minted_volumes r
USING (
  WITH linked AS (SELECT DISTINCT work_id FROM openalex.works.work_volumes WHERE COALESCE(method, '') <> 'minted'),
  loc AS (
    SELECT LOWER(native_id) AS doi, MAX(source_id) AS source_id, MIN(published_date) AS published_date
    FROM openalex.works.locations_w_types
    WHERE provenance = 'crossref' AND native_id_namespace = 'doi'
    GROUP BY 1
  ),
  allp AS (
    SELECT SPLIT(c.doi, '/')[0] AS prefix, k.work_id IS NOT NULL AS is_linked
    FROM openalex.works.crossref_volume_records c
    LEFT JOIN openalex.works.location_work_ids w
      ON w.provenance = 'crossref' AND w.native_id_namespace = 'doi' AND w.native_id = c.doi
    LEFT JOIN linked k ON k.work_id = w.work_id
    WHERE c.cr_type = 'proceedings-article'
  ),
  pfx AS (SELECT prefix FROM allp GROUP BY prefix HAVING COUNT_IF(is_linked) < 0.5 * COUNT(*)),
  p0 AS (
    SELECT c.doi, SPLIT(c.doi, '/')[0] AS prefix, c.year, c.publisher, NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') AS vtitle, NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.event_name, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') AS event,
      NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') AS other_title,
      CAST(REGEXP_EXTRACT(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.event_name, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''), '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])', 1) AS INT) AS ey,
      CONCAT('event-volume:', SPLIT(c.doi, '/')[0], ':', SUBSTR(SHA1(REGEXP_REPLACE(LOWER(CONCAT(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.event_name, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''), ' | ', NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''))), '[^\\p{L}\\p{N}|]', '')), 1, 16)) AS mint_id, w.work_id, l.source_id, l.published_date, s.type AS source_type, k.work_id IS NOT NULL AS is_linked,
      NOT COALESCE(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' AND NOT COALESCE(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])'
      AND c.volume_number IS NULL AS is_own
    FROM openalex.works.crossref_volume_records c
    LEFT JOIN openalex.works.location_work_ids w
      ON w.provenance = 'crossref' AND w.native_id_namespace = 'doi' AND w.native_id = c.doi
    LEFT JOIN linked k ON k.work_id = w.work_id
    LEFT JOIN loc l ON l.doi = c.doi
    LEFT JOIN openalex.sources.sources s ON s.id = l.source_id
    WHERE c.cr_type = 'proceedings-article' AND SPLIT(c.doi, '/')[0] IN ('10.1115') AND NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '^(?i)vol(ume)?[.]? *[0-9]+[a-z]?([^0-9a-z]|$)'
      AND NOT NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?i)vol(ume)?[.]? *[0-9]+[, ]+(no|issue|iss|num|number)[.]? *[0-9]+' AND SIZE(ARRAY_DISTINCT(REGEXP_EXTRACT_ALL(COALESCE(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.event_name, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''), ''), '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])', 1))) = 1
  ),
  e AS (
    SELECT p0.*, ABS(year - ey) <= 1 AS in_year,
      CONCAT(REGEXP_REPLACE(event, '[\\s,;:.]+$', ''), ', ', vtitle) AS title0,
      COUNT(*) OVER (PARTITION BY mint_id, event, vtitle) AS n_t
    FROM p0 JOIN pfx USING (prefix)
  ),
  g AS (
    SELECT mint_id, MAX(ey) AS year, COUNT(*) AS n, COUNT_IF(is_linked) AS n_real, COUNT_IF(NOT is_own) AS n_other_phase,
      COUNT_IF(in_year) AS n_in,
      COUNT(DISTINCT work_id) AS n_works, COUNT(DISTINCT ey) AS n_ey, COUNT(DISTINCT source_id) AS n_src,
      COUNT(DISTINCT LOWER(other_title)) AS n_other, COUNT_IF(source_type IN ('conference', 'book series')) AS n_conf,
      MIN(CASE WHEN YEAR(published_date) = ey THEN published_date END) AS published_date,
      MAX_BY(publisher, STRUCT(n_t, publisher)) AS publisher, MAX(source_id) AS source_id,
      MAX_BY(title0, STRUCT(n_t, title0)) AS title, MAX_BY(vtitle, STRUCT(n_t, vtitle)) AS vtitle
    FROM e GROUP BY mint_id
  ),
  ok AS (
    SELECT mint_id, CAST(NULL AS STRING) AS issn, CAST(NULL AS ARRAY<STRING>) AS series_issns,
      CAST(NULL AS STRING) AS volume_number, title, CAST(year AS INT) AS year,
      COALESCE(published_date, MAKE_DATE(CAST(year AS INT), 1, 1)) AS published_date, publisher, source_id,
      n_in AS papers, vtitle
    FROM g
    WHERE n_real = 0 AND n_other_phase = 0 AND n_conf >= 0.9 * n AND n_works >= 2 AND n_in >= 0.95 * n AND n_ey = 1 AND n_src <= 1
      AND n_other <= 1 AND NULLIF(TRIM(title), '') IS NOT NULL
  ),
  same AS (
    SELECT tk FROM (SELECT REGEXP_REPLACE(LOWER(title), '[^\\p{L}\\p{N}]', '') AS tk, mint_id FROM ok)
    GROUP BY tk HAVING COUNT(DISTINCT mint_id) > 1
  ),
  fresh AS (SELECT ok.mint_id FROM ok LEFT ANTI JOIN openalex.works.minted_volumes m ON m.mint_id = ok.mint_id),
  known AS (
    SELECT REGEXP_REPLACE(LOWER(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '')), '[^\\p{L}\\p{N}]', '') AS tk, publication_year AS y, type
    FROM openalex.works.openalex_works
    WHERE type IN ('book', 'paratext', 'other', 'conference-paper', 'report') AND title IS NOT NULL
    UNION ALL
    SELECT REGEXP_REPLACE(LOWER(title), '[^\\p{L}\\p{N}]', ''), year, 'book' FROM openalex.works.minted_volumes
  ),
  fk AS (  -- the full title, and the bare volume title when it says more than its number
    SELECT ok.mint_id, ok.year, 'full' AS kind, REGEXP_REPLACE(LOWER(ok.title), '[^\\p{L}\\p{N}]', '') AS tk
    FROM ok JOIN fresh USING (mint_id)
    UNION ALL
    SELECT ok.mint_id, ok.year, 'bare', REGEXP_REPLACE(LOWER(ok.vtitle), '[^\\p{L}\\p{N}]', '')
    FROM ok JOIN fresh USING (mint_id)
    WHERE REGEXP_REPLACE(LOWER(ok.vtitle), '^vol(ume)?[.]? *[0-9]+[a-z]?', '') RLIKE '[a-z]'
  ),
  dup AS (
    SELECT DISTINCT fk.mint_id
    FROM fk JOIN known k ON k.tk = fk.tk AND ABS(k.y - fk.year) <= 1
    WHERE fk.kind = 'full' OR k.type IN ('book', 'report', 'other')
  )
  SELECT ok.* EXCEPT (vtitle), fresh.mint_id IS NOT NULL AS is_fresh, dup.mint_id IS NOT NULL OR sm.tk IS NOT NULL AS is_dup
  FROM ok LEFT JOIN fresh USING (mint_id) LEFT JOIN dup USING (mint_id)
  LEFT JOIN same sm ON sm.tk = REGEXP_REPLACE(LOWER(ok.title), '[^\\p{L}\\p{N}]', '')
) s
ON r.mint_id = s.mint_id
WHEN MATCHED AND NOT (r.title <=> s.title AND r.year <=> s.year AND r.published_date <=> s.published_date
    AND r.publisher <=> s.publisher AND r.source_id <=> s.source_id) THEN UPDATE SET
  title = s.title, year = s.year, published_date = s.published_date, publisher = s.publisher, source_id = s.source_id,
  papers = s.papers, last_seen = CURRENT_DATE(), updated_at = CURRENT_DATE()
WHEN MATCHED THEN UPDATE SET papers = s.papers, last_seen = CURRENT_DATE()
WHEN NOT MATCHED AND s.is_fresh AND NOT s.is_dup THEN INSERT (mint_id, issn, series_issns, volume_number, title, year,
  published_date, publisher, source_id, status, papers, first_seen, last_seen, updated_at)
  VALUES (s.mint_id, s.issn, s.series_issns, s.volume_number, s.title, s.year, s.published_date, s.publisher,
  s.source_id, 'active', s.papers, CURRENT_DATE(), CURRENT_DATE(), CURRENT_DATE())
WHEN NOT MATCHED BY SOURCE AND r.mint_id LIKE 'event-volume:%' AND r.last_seen = CURRENT_DATE() THEN UPDATE SET
  last_seen = DATE_SUB(CURRENT_DATE(), 1)

In [0]:
-- oxjob #1528 (22 Nov 2026): proceedings papers that no rule above links take their minted volume (method `minted`):
-- an `active` registry row with the volume number of the paper that shares an ISSN with it, year within 1 of the volume
-- year, and exactly one such row. A work keeps any volume it already has through one of its DOIs, and a work whose DOIs
-- name two different minted volumes takes neither. The W id of the volume comes from its minted location
-- (CreateLocationsWithTypes, provenance `openalex-series`, namespace `volume`), so a volume first registered tonight
-- gets its location, and so its papers, the next night.
INSERT INTO openalex.works.work_volumes
WITH m AS (
  SELECT c.doi, MIN(r.mint_id) AS mint_id, MIN(r.title) AS title
  FROM openalex.works.crossref_volume_records c
  JOIN openalex.works.minted_volumes r
    ON r.status = 'active' AND r.volume_number = c.volume_number AND ARRAYS_OVERLAP(r.series_issns, c.issns)
   AND ABS(c.year - r.year) <= 1
  WHERE c.cr_type = 'proceedings-article'
  GROUP BY c.doi
  HAVING COUNT(DISTINCT r.mint_id) = 1
),
p AS (
  SELECT part.work_id, m.doi, m.mint_id, m.title
  FROM m JOIN openalex.works.location_work_ids part
    ON part.provenance = 'crossref' AND part.native_id_namespace = 'doi' AND part.native_id = m.doi
  WHERE part.work_id IS NOT NULL
),
one AS (SELECT work_id FROM p GROUP BY work_id HAVING COUNT(DISTINCT mint_id) = 1)
SELECT p.work_id, p.doi AS native_id, vol.work_id AS volume_work_id, CAST(NULL AS STRING) AS volume_doi,
  p.title AS volume_display_name, 'minted' AS method
FROM p
JOIN one USING (work_id)
JOIN openalex.works.location_work_ids vol
  ON vol.provenance = 'openalex-series' AND vol.native_id_namespace = 'volume' AND vol.native_id = p.mint_id
LEFT ANTI JOIN (SELECT DISTINCT work_id FROM openalex.works.work_volumes) w ON w.work_id = p.work_id
WHERE vol.work_id IS NOT NULL AND p.work_id <> vol.work_id

In [0]:
-- oxjob #1528 (23 Nov 2026): proceedings papers that no rule above links (phase 1 minted volumes included) take their
-- phase 2 minted volume: the `active` registry row that qualified in the latest run of the registry cell above (its
-- last_seen is the newest: the date of the run, not the clock, so a run that crosses midnight still links) whose mint_id is their DOI prefix plus their edition title, year within 1 of the volume year. A work keeps any
-- volume it already has, and a work whose DOIs name two different minted volumes takes neither. The W id of the volume
-- comes from its minted location, so a volume first registered tonight gets its papers the next night.
-- A paper with no issued year uses its title_year (oxjob #1528, 24 Nov 2026).
INSERT INTO openalex.works.work_volumes
WITH latest AS (SELECT MAX(last_seen) AS d FROM openalex.works.minted_volumes WHERE mint_id LIKE 'edition:%'),
m AS (
  SELECT c.doi, r.mint_id, r.title
  FROM openalex.works.crossref_volume_records c
  JOIN openalex.works.minted_volumes r
    ON r.status = 'active' AND r.mint_id = CONCAT('edition:', SPLIT(c.doi, '/')[0], ':', SUBSTR(SHA1(REGEXP_REPLACE(LOWER(CASE WHEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') WHEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' THEN NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') END), '[^\\p{L}\\p{N}]', '')), 1, 16)) AND ABS(c.title_year - r.year) <= 1
  JOIN latest ON r.last_seen = latest.d
  WHERE c.cr_type = 'proceedings-article' AND r.mint_id LIKE 'edition:%'
),
p AS (
  SELECT part.work_id, m.doi, m.mint_id, m.title
  FROM m JOIN openalex.works.location_work_ids part
    ON part.provenance = 'crossref' AND part.native_id_namespace = 'doi' AND part.native_id = m.doi
  WHERE part.work_id IS NOT NULL
),
one AS (SELECT work_id FROM p GROUP BY work_id HAVING COUNT(DISTINCT mint_id) = 1)
SELECT p.work_id, p.doi AS native_id, vol.work_id AS volume_work_id, CAST(NULL AS STRING) AS volume_doi,
  p.title AS volume_display_name, 'minted' AS method
FROM p
JOIN one USING (work_id)
JOIN openalex.works.location_work_ids vol
  ON vol.provenance = 'openalex-series' AND vol.native_id_namespace = 'volume' AND vol.native_id = p.mint_id
LEFT ANTI JOIN (SELECT DISTINCT work_id FROM openalex.works.work_volumes) w ON w.work_id = p.work_id
WHERE vol.work_id IS NOT NULL AND p.work_id <> vol.work_id

In [0]:
-- oxjob #1528 (24 Nov 2026): proceedings papers that no rule above links (phase 1 and 2 minted volumes included) take
-- their phase 3 minted volume: the `active` registry row that qualified in the latest run of the phase 3 registry cell
-- (its last_seen is the newest among phase 3 rows) whose mint_id is their DOI prefix, event and volume title, year within
-- 1 of the volume year. A work keeps any volume it already has, and a work whose DOIs name two different minted volumes
-- takes neither. The W id comes from the minted location, so a volume first registered tonight gets its papers the next
-- night.
INSERT INTO openalex.works.work_volumes
WITH latest AS (
  SELECT MAX(last_seen) AS d FROM openalex.works.minted_volumes WHERE mint_id LIKE 'event-volume:%' AND status = 'active'),
m AS (
  SELECT c.doi, r.mint_id, r.title
  FROM openalex.works.crossref_volume_records c
  JOIN openalex.works.minted_volumes r
    ON r.status = 'active' AND r.mint_id = CONCAT('event-volume:', SPLIT(c.doi, '/')[0], ':', SUBSTR(SHA1(REGEXP_REPLACE(LOWER(CONCAT(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.event_name, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''), ' | ', NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''))), '[^\\p{L}\\p{N}|]', '')), 1, 16)) AND ABS(c.year - r.year) <= 1
  JOIN latest ON r.last_seen = latest.d
  WHERE c.cr_type = 'proceedings-article' AND SPLIT(c.doi, '/')[0] IN ('10.1115') AND NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '^(?i)vol(ume)?[.]? *[0-9]+[a-z]?([^0-9a-z]|$)'
      AND NOT NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), '') RLIKE '(?i)vol(ume)?[.]? *[0-9]+[, ]+(no|issue|iss|num|number)[.]? *[0-9]+' AND SIZE(ARRAY_DISTINCT(REGEXP_EXTRACT_ALL(COALESCE(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.event_name, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''), ''), '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])', 1))) = 1
      AND NOT COALESCE(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])' AND NOT COALESCE(NULLIF(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(c.container_title2, '&amp;', '&'), '&lt;', '<'), '&gt;', '>'), '&quot;', '"'), '&#39;', "'")), ''), '') RLIKE '(?<![0-9])(19[5-9][0-9]|20[0-3][0-9])(?![0-9])'
      AND c.volume_number IS NULL AND r.mint_id LIKE 'event-volume:%'
),
p AS (
  SELECT part.work_id, m.doi, m.mint_id, m.title
  FROM m JOIN openalex.works.location_work_ids part
    ON part.provenance = 'crossref' AND part.native_id_namespace = 'doi' AND part.native_id = m.doi
  WHERE part.work_id IS NOT NULL
),
one AS (SELECT work_id FROM p GROUP BY work_id HAVING COUNT(DISTINCT mint_id) = 1)
SELECT p.work_id, p.doi AS native_id, vol.work_id AS volume_work_id, CAST(NULL AS STRING) AS volume_doi,
  p.title AS volume_display_name, 'minted' AS method
FROM p
JOIN one USING (work_id)
JOIN openalex.works.location_work_ids vol
  ON vol.provenance = 'openalex-series' AND vol.native_id_namespace = 'volume' AND vol.native_id = p.mint_id
LEFT ANTI JOIN (SELECT DISTINCT work_id FROM openalex.works.work_volumes) w ON w.work_id = p.work_id
WHERE vol.work_id IS NOT NULL AND p.work_id <> vol.work_id